## 10.07 Transformer

### 环境配置

In [1]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
import os
import sys
sys.path.insert(0, "..")
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import pypto
    import math
    import torch
    from torch import nn
    import torch_npu

warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="On the interactive interface")
warnings.filterwarnings("ignore", message="Cannot create tensor")

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()

from src.utils import (Encoder, Decoder, EncoderDecoder, load_data_nmt,
                       train_seq2seq, predict_seq2seq, bleu, sequence_mask)
from src.pypto_ops import (PyPTOBMM, PyPTOSoftmax, PyPTOLinear, PyPTOReLU,
                           PyPTOLayerNorm)

torch.manual_seed(0);

### 练习 10.7.1

**题目：** 在实验中训练更深的Transformer将如何影响训练速度和翻译效果？

**解答：** 增加 `num_layers`（编码器/解码器层数）会带来两方面的变化：

- **训练速度**：层数越多，每个样本的前向/反向计算量线性增大（每层含 2 个多头注意力 + 2 个前馈网络 + 3 次 AddNorm），训练时间近似线性增长；
- **翻译效果**：更深的模型表达能力更强，在数据充足时通常能降低损失、提升 BLEU；但 Transformer 是深网络，过深会加剧优化困难（需要更精细的初始化、学习率调度与残差连接），在小数据集上（如本节 600 句对的英法数据集）甚至可能过拟合或训练不稳定。

下面以 `num_layers=4` 与 `num_layers=2` 对比（训练 200 个迭代周期至收敛，与主 notebook 10.7 节一致）。

以下使用 `torch` 编程进行验证（定义基础组件）：

In [2]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
class PositionWiseFFN(nn.Module):
    def __init__(self, ffn_num_input, ffn_num_hiddens, ffn_num_outputs, **kwargs):
        super().__init__(**kwargs)
        self.dense1 = nn.Linear(ffn_num_input, ffn_num_hiddens)
        self.relu = nn.ReLU()
        self.dense2 = nn.Linear(ffn_num_hiddens, ffn_num_outputs)

    def forward(self, X):
        return self.dense2(self.relu(self.dense1(X)))


class AddNorm(nn.Module):
    def __init__(self, normalized_shape, dropout, **kwargs):
        super().__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(normalized_shape)

    def forward(self, X, Y):
        return self.ln(self.dropout(Y) + X)


def masked_softmax(X, valid_lens):
    if valid_lens is None:
        return nn.functional.softmax(X, dim=-1)
    shape = X.shape
    if valid_lens.dim() == 1:
        valid_lens = torch.repeat_interleave(valid_lens, shape[1])
    else:
        valid_lens = valid_lens.reshape(-1)
    X = sequence_mask(X.reshape(-1, shape[-1]), valid_lens, value=-1e6)
    return nn.functional.softmax(X.reshape(shape), dim=-1)


class DotProductAttention(nn.Module):
    def __init__(self, dropout, **kwargs):
        super().__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values, valid_lens=None):
        d = queries.shape[-1]
        scores = torch.bmm(queries, keys.transpose(1, 2)) / math.sqrt(d)
        self.attention_weights = masked_softmax(scores, valid_lens)
        return torch.bmm(self.dropout(self.attention_weights), values)


def transpose_qkv(X, num_heads):
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)
    X = X.permute(0, 2, 1, 3)
    return X.reshape(-1, X.shape[2], X.shape[3])


def transpose_output(X, num_heads):
    X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
    X = X.permute(0, 2, 1, 3)
    return X.reshape(X.shape[0], X.shape[1], -1)


class MultiHeadAttention(nn.Module):
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 num_heads, dropout, bias=False, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.attention = DotProductAttention(dropout)
        self.W_q = nn.Linear(query_size, num_hiddens, bias=bias)
        self.W_k = nn.Linear(key_size, num_hiddens, bias=bias)
        self.W_v = nn.Linear(value_size, num_hiddens, bias=bias)
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)

    def forward(self, queries, keys, values, valid_lens):
        queries = transpose_qkv(self.W_q(queries), self.num_heads)
        keys = transpose_qkv(self.W_k(keys), self.num_heads)
        values = transpose_qkv(self.W_v(values), self.num_heads)
        if valid_lens is not None:
            valid_lens = torch.repeat_interleave(
                valid_lens, repeats=self.num_heads, dim=0)
        output = self.attention(queries, keys, values, valid_lens)
        output_concat = transpose_output(output, self.num_heads)
        return self.W_o(output_concat)


class PositionalEncoding(nn.Module):
    def __init__(self, num_hiddens, dropout, max_len=1000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.P = torch.zeros((1, max_len, num_hiddens))
        X = torch.arange(max_len, dtype=torch.float32).reshape(
            -1, 1) / torch.pow(10000, torch.arange(
            0, num_hiddens, 2, dtype=torch.float32) / num_hiddens)
        self.P[:, :, 0::2] = torch.sin(X)
        self.P[:, :, 1::2] = torch.cos(X)

    def forward(self, X):
        X = X + self.P[:, :X.shape[1], :].to(X.device)
        return self.dropout(X)


class EncoderBlock(nn.Module):
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 norm_shape, ffn_num_input, ffn_num_hiddens, num_heads,
                 dropout, use_bias=False, **kwargs):
        super().__init__(**kwargs)
        self.attention = MultiHeadAttention(
            key_size, query_size, value_size, num_hiddens, num_heads, dropout,
            use_bias)
        self.addnorm1 = AddNorm(norm_shape, dropout)
        self.ffn = PositionWiseFFN(ffn_num_input, ffn_num_hiddens, num_hiddens)
        self.addnorm2 = AddNorm(norm_shape, dropout)

    def forward(self, X, valid_lens):
        Y = self.addnorm1(X, self.attention(X, X, X, valid_lens))
        return self.addnorm2(Y, self.ffn(Y))


class TransformerEncoder(Encoder):
    def __init__(self, vocab_size, key_size, query_size, value_size,
                 num_hiddens, norm_shape, ffn_num_input, ffn_num_hiddens,
                 num_heads, num_layers, dropout, use_bias=False, **kwargs):
        super().__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.embedding = nn.Embedding(vocab_size, num_hiddens)
        self.pos_encoding = PositionalEncoding(num_hiddens, dropout)
        self.blks = nn.Sequential()
        for i in range(num_layers):
            self.blks.add_module("block"+str(i),
                EncoderBlock(key_size, query_size, value_size, num_hiddens,
                             norm_shape, ffn_num_input, ffn_num_hiddens,
                             num_heads, dropout, use_bias))

    def forward(self, X, valid_lens, *args):
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.num_hiddens))
        self.attention_weights = [None] * len(self.blks)
        for i, blk in enumerate(self.blks):
            X = blk(X, valid_lens)
            self.attention_weights[i] = blk.attention.attention.attention_weights
        return X


class AttentionDecoder(Decoder):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    @property
    def attention_weights(self):
        raise NotImplementedError


class DecoderBlock(nn.Module):
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 norm_shape, ffn_num_input, ffn_num_hiddens, num_heads,
                 dropout, i, **kwargs):
        super().__init__(**kwargs)
        self.i = i
        self.attention1 = MultiHeadAttention(
            key_size, query_size, value_size, num_hiddens, num_heads, dropout)
        self.addnorm1 = AddNorm(norm_shape, dropout)
        self.attention2 = MultiHeadAttention(
            key_size, query_size, value_size, num_hiddens, num_heads, dropout)
        self.addnorm2 = AddNorm(norm_shape, dropout)
        self.ffn = PositionWiseFFN(ffn_num_input, ffn_num_hiddens, num_hiddens)
        self.addnorm3 = AddNorm(norm_shape, dropout)

    def forward(self, X, state):
        enc_outputs, enc_valid_lens = state[0], state[1]
        if state[2][self.i] is None:
            key_values = X
        else:
            key_values = torch.cat((state[2][self.i], X), axis=1)
        state[2][self.i] = key_values
        if self.training:
            batch_size, num_steps, _ = X.shape
            dec_valid_lens = torch.arange(
                1, num_steps + 1, device=X.device).repeat(batch_size, 1)
        else:
            dec_valid_lens = None
        X2 = self.attention1(X, key_values, key_values, dec_valid_lens)
        Y = self.addnorm1(X, X2)
        Y2 = self.attention2(Y, enc_outputs, enc_outputs, enc_valid_lens)
        Z = self.addnorm2(Y, Y2)
        return self.addnorm3(Z, self.ffn(Z)), state


class TransformerDecoder(AttentionDecoder):
    def __init__(self, vocab_size, key_size, query_size, value_size,
                 num_hiddens, norm_shape, ffn_num_input, ffn_num_hiddens,
                 num_heads, num_layers, dropout, **kwargs):
        super().__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, num_hiddens)
        self.pos_encoding = PositionalEncoding(num_hiddens, dropout)
        self.blks = nn.Sequential()
        for i in range(num_layers):
            self.blks.add_module("block"+str(i),
                DecoderBlock(key_size, query_size, value_size, num_hiddens,
                             norm_shape, ffn_num_input, ffn_num_hiddens,
                             num_heads, dropout, i))
        self.dense = nn.Linear(num_hiddens, vocab_size)

    def init_state(self, enc_outputs, enc_valid_lens, *args):
        return [enc_outputs, enc_valid_lens, [None] * self.num_layers]

    def forward(self, X, state):
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.num_hiddens))
        self._attention_weights = [[None] * len(self.blks) for _ in range(2)]
        for i, blk in enumerate(self.blks):
            X, state = blk(X, state)
            self._attention_weights[0][i] = blk.attention1.attention.attention_weights
            self._attention_weights[1][i] = blk.attention2.attention.attention_weights
        return self.dense(X), state

    @property
    def attention_weights(self):
        return self._attention_weights

In [3]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
def make_transformer(num_layers):
    num_hiddens, dropout, batch_size, num_steps = 32, 0.1, 64, 10
    ffn_num_input, ffn_num_hiddens, num_heads = 32, 64, 4
    key_size, query_size, value_size = 32, 32, 32
    norm_shape = [32]
    train_iter, src_vocab, tgt_vocab = load_data_nmt(batch_size, num_steps)
    encoder = TransformerEncoder(
        len(src_vocab), key_size, query_size, value_size, num_hiddens,
        norm_shape, ffn_num_input, ffn_num_hiddens, num_heads,
        num_layers, dropout)
    decoder = TransformerDecoder(
        len(tgt_vocab), key_size, query_size, value_size, num_hiddens,
        norm_shape, ffn_num_input, ffn_num_hiddens, num_heads,
        num_layers, dropout)
    return EncoderDecoder(encoder, decoder), train_iter, src_vocab, tgt_vocab, num_steps


# 说明：train_seq2seq 内部 Animator 每 10 个 epoch 调用 clear_output(wait=True)，
# 同一 cell 内后续训练会清除前面训练的打印输出，因此把两种层数拆成两个 cell 执行。
net, train_iter, src_vocab, tgt_vocab, num_steps = make_transformer(2)
train_seq2seq(net, train_iter, lr=0.005, num_epochs=200, tgt_vocab=tgt_vocab,
              device=device)
translation, _ = predict_seq2seq(net, 'go .', src_vocab, tgt_vocab,
                                 num_steps, device)
print(f'num_layers=2: go . => {translation}, '
      f'bleu {bleu(translation, "va !", k=2):.3f}')


loss 0.033, 6156.8 tokens/sec on npu:0
num_layers=2: go . => va !, bleu 1.000


In [4]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
# 练习 10.7.1（续）：num_layers=4 对比（训练时间约为 2 层的 2 倍）
net, train_iter, src_vocab, tgt_vocab, num_steps = make_transformer(4)
train_seq2seq(net, train_iter, lr=0.005, num_epochs=200, tgt_vocab=tgt_vocab,
              device=device)
translation, _ = predict_seq2seq(net, 'go .', src_vocab, tgt_vocab,
                                 num_steps, device)
print(f'num_layers=4: go . => {translation}, '
      f'bleu {bleu(translation, "va !", k=2):.3f}')


loss 0.182, 3925.6 tokens/sec on npu:0
num_layers=4: go . => <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk>, bleu 0.000


使用 `PyPTO` 编程进行验证（将 `nn.Linear` / `nn.ReLU` / `nn.LayerNorm` / `torch.bmm` / `F.softmax` 替换为 `PyPTOLinear` / `PyPTOReLU` / `PyPTOLayerNormModule` / `PyPTOBMM` / `PyPTOSoftmax`，注意力组件直接复用 `src.attention` ）：

In [5]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
from src.pypto_ops import (PyPTOLinear, PyPTOReLU, PyPTOLayerNormModule)
from src.attention import (masked_softmax, DotProductAttention,
                           MultiHeadAttention, PositionalEncoding,
                           AttentionDecoder)


class PositionWiseFFNPypto(nn.Module):
    def __init__(self, ffn_num_input, ffn_num_hiddens, ffn_num_outputs,
                 **kwargs):
        super().__init__(**kwargs)
        self.dense1 = PyPTOLinear(ffn_num_input, ffn_num_hiddens)
        self.relu = PyPTOReLU()
        self.dense2 = PyPTOLinear(ffn_num_hiddens, ffn_num_outputs)

    def forward(self, X):
        return self.dense2(self.relu(self.dense1(X)))


class AddNormPypto(nn.Module):
    def __init__(self, normalized_shape, dropout, **kwargs):
        super().__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)
        self.ln = PyPTOLayerNormModule(normalized_shape)

    def forward(self, X, Y):
        return self.ln(self.dropout(Y) + X)


class EncoderBlockPypto(nn.Module):
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 norm_shape, ffn_num_input, ffn_num_hiddens, num_heads,
                 dropout, use_bias=False, **kwargs):
        super().__init__(**kwargs)
        self.attention = MultiHeadAttention(
            key_size, query_size, value_size, num_hiddens, num_heads, dropout,
            use_bias)
        self.addnorm1 = AddNormPypto(norm_shape, dropout)
        self.ffn = PositionWiseFFNPypto(
            ffn_num_input, ffn_num_hiddens, num_hiddens)
        self.addnorm2 = AddNormPypto(norm_shape, dropout)

    def forward(self, X, valid_lens):
        Y = self.addnorm1(X, self.attention(X, X, X, valid_lens))
        return self.addnorm2(Y, self.ffn(Y))


class TransformerEncoderPypto(Encoder):
    def __init__(self, vocab_size, key_size, query_size, value_size,
                 num_hiddens, norm_shape, ffn_num_input, ffn_num_hiddens,
                 num_heads, num_layers, dropout, use_bias=False, **kwargs):
        super().__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.embedding = nn.Embedding(vocab_size, num_hiddens)
        self.pos_encoding = PositionalEncoding(num_hiddens, dropout)
        self.blks = nn.Sequential()
        for i in range(num_layers):
            self.blks.add_module("block" + str(i),
                EncoderBlockPypto(key_size, query_size, value_size,
                                  num_hiddens, norm_shape, ffn_num_input,
                                  ffn_num_hiddens, num_heads, dropout,
                                  use_bias))

    def forward(self, X, valid_lens, *args):
        X = self.pos_encoding(
            self.embedding(X) * math.sqrt(self.num_hiddens))
        self.attention_weights = [None] * len(self.blks)
        for i, blk in enumerate(self.blks):
            X = blk(X, valid_lens)
            self.attention_weights[i] = blk.attention.attention.attention_weights
        return X


class DecoderBlockPypto(nn.Module):
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 norm_shape, ffn_num_input, ffn_num_hiddens, num_heads,
                 dropout, i, **kwargs):
        super().__init__(**kwargs)
        self.i = i
        self.attention1 = MultiHeadAttention(
            key_size, query_size, value_size, num_hiddens, num_heads, dropout)
        self.addnorm1 = AddNormPypto(norm_shape, dropout)
        self.attention2 = MultiHeadAttention(
            key_size, query_size, value_size, num_hiddens, num_heads, dropout)
        self.addnorm2 = AddNormPypto(norm_shape, dropout)
        self.ffn = PositionWiseFFNPypto(
            ffn_num_input, ffn_num_hiddens, num_hiddens)
        self.addnorm3 = AddNormPypto(norm_shape, dropout)

    def forward(self, X, state):
        enc_outputs, enc_valid_lens = state[0], state[1]
        if state[2][self.i] is None:
            key_values = X
        else:
            key_values = torch.cat((state[2][self.i], X), axis=1)
        state[2][self.i] = key_values
        if self.training:
            batch_size, num_steps, _ = X.shape
            dec_valid_lens = torch.arange(
                1, num_steps + 1, device=X.device).repeat(batch_size, 1)
        else:
            dec_valid_lens = None
        X2 = self.attention1(X, key_values, key_values, dec_valid_lens)
        Y = self.addnorm1(X, X2)
        Y2 = self.attention2(Y, enc_outputs, enc_outputs, enc_valid_lens)
        Z = self.addnorm2(Y, Y2)
        return self.addnorm3(Z, self.ffn(Z)), state


class TransformerDecoderPypto(AttentionDecoder):
    def __init__(self, vocab_size, key_size, query_size, value_size,
                 num_hiddens, norm_shape, ffn_num_input, ffn_num_hiddens,
                 num_heads, num_layers, dropout, **kwargs):
        super().__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, num_hiddens)
        self.pos_encoding = PositionalEncoding(num_hiddens, dropout)
        self.blks = nn.Sequential()
        for i in range(num_layers):
            self.blks.add_module("block" + str(i),
                DecoderBlockPypto(key_size, query_size, value_size,
                                  num_hiddens, norm_shape, ffn_num_input,
                                  ffn_num_hiddens, num_heads, dropout, i))
        self.dense = PyPTOLinear(num_hiddens, vocab_size)

    def init_state(self, enc_outputs, enc_valid_lens, *args):
        return [enc_outputs, enc_valid_lens, [None] * self.num_layers]

    def forward(self, X, state):
        X = self.pos_encoding(
            self.embedding(X) * math.sqrt(self.num_hiddens))
        self._attention_weights = [[None] * len(self.blks) for _ in range(2)]
        for i, blk in enumerate(self.blks):
            X, state = blk(X, state)
            self._attention_weights[0][i] = (
                blk.attention1.attention.attention_weights)
            self._attention_weights[1][i] = (
                blk.attention2.attention.attention_weights)
        return self.dense(X), state

    @property
    def attention_weights(self):
        return self._attention_weights


In [6]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
# 说明：train_seq2seq 内部 Animator 每 10 个 epoch 调用 clear_output(wait=True)，
# 同一 cell 内后续训练会清除前面训练的打印输出，因此把两种层数拆成两个 cell 执行。
def make_transformer_pypto(num_layers):
    num_hiddens, dropout, batch_size, num_steps = 32, 0.1, 64, 10
    ffn_num_input, ffn_num_hiddens, num_heads = 32, 64, 4
    key_size, query_size, value_size = 32, 32, 32
    norm_shape = [32]
    train_iter, src_vocab, tgt_vocab = load_data_nmt(batch_size, num_steps)
    encoder = TransformerEncoderPypto(
        len(src_vocab), key_size, query_size, value_size, num_hiddens,
        norm_shape, ffn_num_input, ffn_num_hiddens, num_heads,
        num_layers, dropout)
    decoder = TransformerDecoderPypto(
        len(tgt_vocab), key_size, query_size, value_size, num_hiddens,
        norm_shape, ffn_num_input, ffn_num_hiddens, num_heads,
        num_layers, dropout)
    return (EncoderDecoder(encoder, decoder), train_iter, src_vocab,
            tgt_vocab, num_steps)


net, train_iter, src_vocab, tgt_vocab, num_steps = make_transformer_pypto(2)
train_seq2seq(net, train_iter, lr=0.005, num_epochs=200, tgt_vocab=tgt_vocab,
              device=device)
translation, _ = predict_seq2seq(net, 'go .', src_vocab, tgt_vocab,
                                 num_steps, device)
print(f'num_layers=2 (PyPTO): go . => {translation}, '
      f'bleu {bleu(translation, "va !", k=2):.3f}')


loss 0.032, 3672.4 tokens/sec on npu:0


num_layers=2 (PyPTO): go . => va !, bleu 1.000


In [7]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
# 练习 10.7.1（续）：num_layers=4 对比（PyPTO 版，训练时间约为 2 层的 2 倍）
net, train_iter, src_vocab, tgt_vocab, num_steps = make_transformer_pypto(4)
train_seq2seq(net, train_iter, lr=0.005, num_epochs=200, tgt_vocab=tgt_vocab,
              device=device)
translation, _ = predict_seq2seq(net, 'go .', src_vocab, tgt_vocab,
                                 num_steps, device)
print(f'num_layers=4 (PyPTO): go . => {translation}, '
      f'bleu {bleu(translation, "va !", k=2):.3f}')


loss 0.092, 2192.1 tokens/sec on npu:0
num_layers=4 (PyPTO): go . => <unk> ., bleu 0.000


### 练习 10.7.2

**题目：** 在Transformer中使用加性注意力取代缩放点积注意力是不是个好办法？为什么？

**解答：** 通常不是好办法。原因：

- **计算效率**：缩放点积注意力只需两次批量矩阵乘法（$QK^\top$ 与 $\mathrm{softmax}(\cdot)V$），充分利用矩阵乘法的硬件加速；加性注意力需要 $W_q, W_k, w_v$ 三次线性变换 + 广播求和 + `tanh`，计算量更大；
- **参数开销**：加性注意力每个头多三组可学习参数，多头（如 8 头）叠加后参数量明显增加；
- **关键前提**：Transformer 中查询与键同维（都是 `num_hiddens`），满足点积注意力的使用条件，因此可以享受其高效性。

结论：在查询键同维的场景下优先使用缩放点积注意力；加性注意力主要用于查询与键长度不同的场景（如 [10.4 节](../10.04_bahdanau_attention.ipynb)之前讨论的情形）。

### 练习 10.7.3

**题目：** 对于语言模型，应该使用Transformer的编码器还是解码器，或者两者都用？如何设计？

**解答：** 取决于任务类型：

- **双向上下文（BERT 风格）**：使用编码器。语言建模目标为"完形填空"（masked LM），每个位置的表示可以看到整个序列（含未来词元），训练效率高；
- **自回归生成（GPT 风格）**：使用解码器。逐词元生成时必须保持自回归属性（掩蔽自注意力保证查询只看过去），用于文本生成、对话等；
- **两者都用（编码器-解码器）**：用于序列转换任务（机器翻译、摘要），编码器读全文，解码器在编码信息指导下生成。

设计要点：语言模型通常堆叠解码器层（含掩蔽自注意力 + 位置编码），输出层预测词表分布；若只做表示学习则用编码器 + 可选任务头。

### 练习 10.7.4

**题目：** 如果输入序列很长，Transformer会面临什么挑战？为什么？

**解答：** 主要挑战是**二次方复杂度**：自注意力需要计算 $n \times n$ 的注意力分数矩阵并存储，时间和空间复杂度均为 $\mathcal{O}(n^2)$（$n$ 为序列长度）。

- **计算**：每层的计算量为 $\mathcal{O}(n^2 d)$（$d$ 为特征维度），序列长度翻倍则计算量翻四倍；
- **显存**：$n \times n$ 的分数矩阵与注意力权重矩阵随 $n$ 平方增长，长序列（如整本书、长文档、高分辨率图像）很容易耗尽显存；
- 因此对长序列需要稀疏注意力、滑动窗口注意力、低秩近似（Linformer）、FlashAttention 等改进（见练习 10.7.5）。

### 练习 10.7.5

**题目：** 如何提高Transformer的计算速度和内存使用效率？提示：可以参考论文 ([Tay et al., 2020](https://zh.d2l.ai/chapter_references/zreferences.html#Tay.Dehghani.Bahri.ea.2020))。

**解答：** 可从以下方向入手：

- **稀疏注意力（Sparse Attention）**：限制每个查询只关注局部窗口或特定步长（stride）的位置（如 Longformer、BigBird），将复杂度降为 $\mathcal{O}(n)$ 或 $\mathcal{O}(n \log n)$；
- **低秩/核近似（Low-rank / Kernel Approximation）**：用低秩投影近似 $QK^\top$（Linformer：$\mathcal{O}(n)$），或用核方法近似 softmax（Performer：随机特征映射）；
- **FlashAttention**：将 $QK^\top$ 分块计算并在线 softmax，避免物化 $n \times n$ 矩阵，显著降低显存占用并利用 SRAM 带宽；
- **序列压缩/池化**：对长序列先做局部聚合（如文本中的 segment pooling、图像中的 patch 化），缩短有效序列长度；
- **混合模型**：浅层用局部操作（卷积/RNN），深层用全局注意力。

### 练习 10.7.6

**题目：** 如果不使用卷积神经网络，如何设计基于Transformer模型的图像分类任务？提示：可以参考Vision Transformer ([Dosovitskiy et al., 2021](https://zh.d2l.ai/chapter_references/zreferences.html#Dosovitskiy.Beyer.Kolesnikov.ea.2021))。

**解答：** 参考 Vision Transformer（ViT）的设计：

1. **图像分块（Patch Embedding）**：将 $H \times W \times 3$ 图像切成 $\frac{HW}{P^2}$ 个 $P \times P \times 3$ 的小块，每个小块展平后经线性投影得到 $d$ 维嵌入；
2. **位置编码**：为每个 patch 嵌入加上可学习的位置编码，保留空间顺序信息；
3. **类别标记（CLS token）**：在序列头部拼接一个可学习的 `[CLS]` 标记，最终取其输出表示用于分类；
4. **Transformer 编码器**：堆叠多层标准编码器块（多头自注意力 + 前馈网络 + AddNorm）；
5. **分类头**：对 `[CLS]` 输出做线性分类。

相比 CNN，ViT 没有卷积的归纳偏置（局部性/平移等变性），需要更多数据或更强的正则（数据增强、蒸馏）才能训练好。

---
## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)